In [5]:
import subprocess
import os
from dotenv import load_dotenv

In [2]:
from hcloud import Client

In [6]:
load_dotenv()

True

In [3]:
def create_hetzner_client(keyname: str):
    """Loads key from .env file of given keyname"""
    try:
        hkey = os.environ[keyname]
    except KeyError as e:
        raise KeyError(f"The key {e} does not exist")

    return Client(token=hkey)


In [7]:
cli = create_hetzner_client('HETZNER_API_KEY')

In [8]:
cli.servers.get_all()

In [11]:
svr = cli.servers.get_by_name("tps-server")

In [13]:
svr.public_net.ipv4.ip

'188.245.247.54'

In [2]:
def check_cloud_init(ip):
    cmd = f"ssh -i ~/.ssh/sledge_wsl -o ConnectTimeout=5 -o StrictHostKeyChecking=no ubuntu@{ip} 'cloud-init status'"
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=10)
        return result.stdout.strip() if result.returncode == 0 else result.stderr.strip()
    except: return "Oh, no"

In [3]:
ip = "188.245.215.177"
cmd = f"ssh -i ~/.ssh/sledge_wsl -o ConnectTimeout=5 -o StrictHostKeyChecking=no ubuntu@{ip} 'cloud-init status'"

subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=10)

CompletedProcess(args="ssh -i ~/.ssh/sledge_wsl -o ConnectTimeout=5 -o StrictHostKeyChecking=no ubuntu@188.245.215.177 'cloud-init status'", returncode=0, stdout='status: running\n', stderr='')

In [10]:
import subprocess, time

ip = "188.245.215.177"
cmd = f"ssh -i ~/.ssh/sledge_wsl -o ConnectTimeout=10 -o StrictHostKeyChecking=no ubuntu@{ip} 'cloud-init status'"
for i in range(10):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=20, check=True)
        print(r.stdout.strip())
        break
    except subprocess.TimeoutExpired:
        print(f"timeout {i+1}/10")
    except subprocess.CalledProcessError as e:
        print("err:", e.stderr.strip())
    time.sleep(10)
else:
    raise RuntimeError("host unreachable / cloud-init still booting")

err: 
err: 
err: 
err: 


KeyboardInterrupt: 

In [15]:
import subprocess, time

def check_cloud_init(ip, attempts=6, sleep=5):
    cmd = (
        f"ssh -i ~/.ssh/sledge_wsl "
        f"-o ConnectTimeout=7 -o StrictHostKeyChecking=no "
        f"ubuntu@{ip} "
        f"'cloud-init status --wait'"
    )
    for i in range(attempts):
        try:
            r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=330)
            print("stdout:", r.stdout.strip())
            print("stderr:", r.stderr.strip())
            print("rc:", r.returncode)
            if r.returncode == 0:
                return r.stdout.strip()
            # rc=1 -> still running; rc=2 -> failed
        except subprocess.TimeoutExpired:
            print(f"attempt {i+1}/{attempts}: ssh command timed out")
        except KeyboardInterrupt:
            raise
        except Exception as e:
            print("ssh failed:", type(e).__name__, e)

        if i < attempts-1:
            time.sleep(sleep)

    raise RuntimeError("cloud-init not complete after retries")

print(check_cloud_init("188.245.215.177"))

stdout: 
stderr: ssh: connect to host 188.245.215.177 port 22: Connection timed out
rc: 255
stdout: 
stderr: ssh: connect to host 188.245.215.177 port 22: Connection timed out
rc: 255
stdout: 
stderr: ssh: connect to host 188.245.215.177 port 22: Connection timed out
rc: 255
stdout: 
stderr: ssh: connect to host 188.245.215.177 port 22: Connection timed out
rc: 255
stdout: 
stderr: ssh: connect to host 188.245.215.177 port 22: Connection timed out
rc: 255
stdout: 
stderr: ssh: connect to host 188.245.215.177 port 22: Connection timed out
rc: 255


RuntimeError: cloud-init not complete after retries

In [9]:
check_cloud_init("188.245.215.177")

''